## Imports and config

### Make sure the paths in the config are correct

In [ ]:
import os
import os.path as osp
import sys
module_path = os.path.abspath(os.path.join('../..'))
sys.path.append(module_path)
from src.utils.db import DatabaseManager
from src.utils.process_config import load_config
from itables import show
import pandas as pd

import numpy as np

config_path = os.path.join(module_path, 'src', 'config', 'edit_192_external.json')

class Args:
    def __init__(self, config):
        self.config = config
args = Args(config=config_path)
config = load_config(args.config)
dbm = DatabaseManager(config)
print("Database manager initialized from config:", config_path)
dbm.print_db_schema_counts()





: 

In [ ]:


# columns to show in the leaderboard
score_columns = [
    "chamfer dist",
    "iou",
    "dino v2 sim",
    "validity",
    "instruction",
    "quality",
    "acceptance",
]

# users to show in the leaderboard
users = [
    "human baseline",
    "qwen3.6-27b_cadquery-script",
    "qwen3.6-27b_cadquery-mcp",
    "gemma4-31b_cadquery-script",
    "gemma4-31b_cadquery-mcp",
    "gpt-5.6-luna_cadquery-script",
    "gpt-5.6-luna_cadquery-mcp"
]

LLM_EVAL_RATER = "claude-opus_agentic-eval-rating"


all_edits = {}
for edt in dbm.edits.find({}):
    all_edits[edt["_id"]] = {}

for rating in dbm.ratings.find({}):
    edit_id = rating["edit"]
    ignore_keys = ["_id", "edit", "user", 'result_info', 'job_info', 'submission_time', 'comments', 'flags']
    existing_edit = all_edits.get(edit_id, {})
    
    rating_user = rating["user"]

    existing_rating_user = existing_edit.get(rating_user, {})

    # only add keys that are not in ignore_keys
    for k, v in rating.items():
        if k not in ignore_keys:
            existing_rating_user[k] = v

    #update the edit's ratings with the new rating
    existing_edit[rating_user] = existing_rating_user
    all_edits[edit_id] = existing_edit

def process_single_rating(rating):
    # instruction and qualitiy human/vlm ratings have different defaults for doing nothing/failed runs according to the scoring rubric.
    
    new_rating = {}
    similarity_eval = rating.get("similarity_eval", {})
    new_rating["dino v2 sim"] = similarity_eval.get("dino similarity gt", np.nan)
    new_rating["chamfer dist"] = 1.0 / (similarity_eval.get("chamfer similarity gt", np.nan) + 1e-8)
    new_rating["iou"] = similarity_eval.get("iou gt", np.nan)


    opus_dict = rating.get(LLM_EVAL_RATER, {})
    new_rating["instruction"] = (opus_dict.get("score_instr", 2) - 1 ) / 6.0
    new_rating["quality"] = (opus_dict.get("score_quality", 1) - 1 ) / 6.0


    instruction_threshold = (5-1) / 6.0
    quality_threshold = (5-1) / 6.0

    new_rating["acceptance"] = float(new_rating["instruction"] >= instruction_threshold and new_rating["quality"] >= quality_threshold)
    return new_rating


for k, v in all_edits.items():
    new_rating = process_single_rating(v)
    new_rating["edit_id"] = k
    new_rating["request_id"] = dbm.edits.find_one({"_id": k})["request"]

    edit_user_id = dbm.edits.find_one({"_id": k})["user"]
    request_user_id = dbm.requests.find_one({"_id": new_rating["request_id"]})["user"]
    edit_user = dbm.users.find_one({"_id": edit_user_id})
    new_rating["modality"] = dbm.requests.find_one({"_id": new_rating["request_id"]}).get("modality")

    if edit_user["is_human"]:
        if edit_user_id == request_user_id:
            new_rating["edit_user"] = "human requestor"
        else:
            new_rating["edit_user"] = "human baseline"
    else:
        new_rating["edit_user"] = edit_user_id

    # validity entry - checks that there is a toprightiso, as that means the preprocessing ran OK
    brep_id = dbm.edits.find_one({"_id": k})["brep_end"]
    images = dbm.get_brep_images(brep_id, views=["toprightiso"], format=["jpg", "png"])
    if images is not None and len(images) > 0:
        new_rating["validity"] = 1
    else:
        new_rating["validity"] = 0

    failed_run = bool(
        dbm.edits.find_one({"_id": k}).get("failed_run", False)
    )
    if failed_run:
        new_rating["validity"] = 0
    
    difficulty = dbm.requests.find_one({"_id": new_rating["request_id"]}).get("difficulty")
    new_rating["difficulty"] = difficulty

    all_edits[k] = new_rating

# turn into dataframe, with edit_id as index
df = pd.DataFrame.from_dict(all_edits, orient='index')

# Filter to only the specified users
df_filtered = df[df["edit_user"].isin(users)].copy()
df_melted = df_filtered.melt(id_vars=["edit_user"], value_vars=score_columns, var_name="Metric", value_name="Score")
df_grouped = df_melted.groupby(["edit_user", "Metric"], as_index=False)["Score"].mean()

# Enforce the desired user order and score order
df_grouped["edit_user"] = pd.Categorical(df_grouped["edit_user"], categories=users, ordered=True)
df_grouped["Metric"] = pd.Categorical(df_grouped["Metric"], categories=score_columns, ordered=True)
df_grouped = df_grouped.sort_values(["edit_user", "Metric"])

df_pivot_scores = df_grouped.pivot(index="edit_user", columns="Metric", values="Score")
df_pivot_scores = df_pivot_scores.reindex(index=users, columns=score_columns)

# set all values to 2 decimal places
df_pivot_scores = df_pivot_scores.round(2)

# Print results as tab-separated table for pasting into spreadsheet
# Print header
print("\t".join(["edit_user"] + score_columns))
# Print rows
for user in users:
    row_values = [f"{df_pivot_scores.loc[user, col]:.4f}" for col in score_columns]
    print("\t".join([user] + row_values))

# print results in nice table using itables
show(df_pivot_scores)





In [ ]:
table_end_to_end = (
    df_filtered
    .groupby("edit_user")
    .agg(
        validity=("validity", "mean"),
        instruction=("instruction", "mean"),
        quality=("quality", "mean"),
        acceptance=("acceptance", "mean"),
    )
    .reindex(users)
    .round(3)
)
table_end_to_end[["validity", "acceptance"]] *= 100
table_end_to_end = table_end_to_end.round(2)
table_end_to_end = table_end_to_end.rename(columns={
    "validity": "validity (%)",
    "acceptance": "acceptance (%)",
})

show(table_end_to_end)

In [ ]:
df_valid = df_filtered[df_filtered["validity"] == 1]

table_valid_only = (
    df_valid
    .groupby("edit_user")
    .agg(
        n_valid=("edit_id", "count"),
        chamfer_dist=("chamfer dist", "median"),
        iou=("iou", "median"),
        dino_v2=("dino v2 sim", "median"),
    )
    .reindex(users)
    .round(2)
)

show(table_valid_only)

In [ ]:
from matplotlib.table import table


def paired_table(df, pair_users):
    pair_df = df[df["edit_user"].isin(pair_users)]

    # Requests finden, bei denen beide Arme validity == 1 haben
    valid_counts = (
        pair_df[pair_df["validity"] == 1]
        .groupby("request_id")["edit_user"]
        .nunique()
    )

    shared_requests = valid_counts[
        valid_counts == len(pair_users)
    ].index

    # Nur die gemeinsame gültige Schnittmenge behalten
    paired_df = pair_df[
        pair_df["request_id"].isin(shared_requests)
        & (pair_df["validity"] == 1)
    ]

    metrics = [
        column for column in score_columns
        if column != "validity"
    ]

    table = (
        paired_df
        .groupby("edit_user")[metrics]
        .mean()
        .reindex(pair_users)
        .round(3)
    )
    table["acceptance"] *= 100
    table["n_tasks"] = len(shared_requests)

    table = table.rename(columns={
        "acceptance": "acceptance (%)",
    })

    return table

qwen_users = [
    "qwen3.6-27b_cadquery-script",
    "qwen3.6-27b_cadquery-mcp",
]

qwen_table = paired_table(df, qwen_users)
show(qwen_table)

gemma_users = [
    "gemma4-31b_cadquery-script",
    "gemma4-31b_cadquery-mcp",
]

gemma_table = paired_table(df, gemma_users)
show(gemma_table)

luna_users = [
    "gpt-5.6-luna_cadquery-script",
    "gpt-5.6-luna_cadquery-mcp",
]

luna_table = paired_table(df, luna_users)
show(luna_table)


In [ ]:

print("Mean over all tasks:")
print(table_end_to_end.to_markdown(floatfmt=".2f"))
print()
print("Median over succesful tasks:")
print(table_valid_only.to_markdown(floatfmt=".2f"))
print()
print("Pairweise comparison(Mean)")
print(qwen_table.to_markdown(floatfmt=".2f"))
print()
print(gemma_table.to_markdown(floatfmt=".2f"))
print()
print(luna_table.to_markdown(floatfmt=".2f"))

## Token usage and API cost

Totals include only runs with recorded `token_counts`. Luna cost uses the [official GPT-5.6 Luna prices](https://developers.openai.com/api/docs/models/gpt-5.6-luna) as of 2026-09-11: $0.20/M uncached input tokens, $0.02/M cached input tokens, and $1.20/M output tokens. The stored runs do not contain `cache_write_tokens` or per-call usage, so cache-write charges and the >272K-token per-request surcharge cannot be reconstructed.

In [ ]:
token_model_names = {
    "qwen3.6-27b": "Qwen",
    "gemma4-31b": "Gemma",
    "gpt-5.6-luna": "Luna",
}

# USD per million tokens.
# Source: https://developers.openai.com/api/docs/models/gpt-5.6-luna
api_prices = {
    "gpt-5.6-luna": {
        "input": 0.20,
        "cached_input": 0.02,
        "output": 1.20,
    },
}

token_rows = []
for user in users:
    if user == "human baseline":
        continue

    model_id, harness = user.split("_cadquery-", maxsplit=1)
    arm = "Baseline" if harness == "script" else "Agentic"
    edit_docs = list(dbm.edits.find({"user": user}))
    usage = [
        edit["token_counts"]
        for edit in edit_docs
        if isinstance(edit.get("token_counts"), dict)
    ]

    def token_sum(key):
        return sum(float(item.get(key, 0) or 0) for item in usage)

    input_tokens = token_sum("input_tokens")
    output_tokens = token_sum("output_tokens")
    total_tokens = token_sum("total_tokens")
    has_cache_data = any("cached_input_tokens" in item for item in usage)
    cached_input_tokens = token_sum("cached_input_tokens")

    prices = api_prices.get(model_id)
    uncached_input_tokens = max(input_tokens - cached_input_tokens, 0)
    api_cost = (
        (uncached_input_tokens * prices["input"])
        + (cached_input_tokens * prices["cached_input"])
        + (output_tokens * prices["output"])
    ) / 1e6 if prices and usage else np.nan

    token_rows.append({
        "model": token_model_names.get(model_id, model_id),
        "arm": arm,
        "runs with usage": len(usage),
        "input tokens (M)": input_tokens / 1e6 if usage else np.nan,
        "output tokens (M)": output_tokens / 1e6 if usage else np.nan,
        "total tokens (M)": total_tokens / 1e6 if usage else np.nan,
        "cache-adjusted API cost (USD)": api_cost,
        "cached input (M)": (
            cached_input_tokens / 1e6 if has_cache_data else np.nan
        ),
        "cache hit rate (%)": (
            100 * cached_input_tokens / input_tokens
            if has_cache_data and input_tokens else np.nan
        ),
    })

token_cost_table = (
    pd.DataFrame(token_rows)
    .set_index(["model", "arm"])
    .round(2)
)

show(token_cost_table)
print(token_cost_table.to_markdown(floatfmt=".2f"))


In [ ]:
import matplotlib.pyplot as plt


plot_users = users


display_names = {
    "human baseline": "Human baseline",
    "qwen3.6-27b_cadquery-script": "Qwen Script",
    "qwen3.6-27b_cadquery-mcp": "Qwen MCP",
    "gemma4-31b_cadquery-script": "Gemma Script",
    "gemma4-31b_cadquery-mcp": "Gemma MCP",
    "gpt-5.6-luna_cadquery-script": "Luna Script",
    "gpt-5.6-luna_cadquery-mcp": "Luna MCP",
}

plot_data = df[df["edit_user"].isin(plot_users)].copy()

modality_order = [
    "text",
    "interactive",
    "draw-temp",
    "draw-static",
]

difficulty_order = [
    "easy",
    "medium",
    "hard",
]

modality_colors = [
    "#252525",
    "#E5AA3D",
    "#F48A8A",
    "#E83C3C",
]

difficulty_colors = [
    "#72AF78",
    "#E5AA3D",
    "#DF7C7C",
]
acceptance_by_modality = (
    plot_data
    .groupby(["edit_user", "modality"])["acceptance"]
    .mean()
    .unstack("modality")
    .reindex(index=plot_users, columns=modality_order)
)

acceptance_by_difficulty = (
    plot_data
    .groupby(["edit_user", "difficulty"])["acceptance"]
    .mean()
    .unstack("difficulty")
    .reindex(index=plot_users, columns=difficulty_order)
)

In [ ]:
model_labels = [
    display_names.get(user, user)
    for user in plot_users
]

ax = acceptance_by_modality.plot.bar(
    figsize=(9, 5),
    color=modality_colors,
    width=0.8,
)

ax.set_ylim(0, 1)
ax.set_xlabel("")
ax.set_ylabel("LLM acceptance")
ax.set_title("Acceptance by modality")
ax.set_xticklabels(model_labels, rotation=0)

ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(title="modality", loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
ax = acceptance_by_difficulty.plot.bar(
    figsize=(9, 5),
    color=difficulty_colors,
    width=0.8,
)

ax.set_ylim(0, 1)
ax.set_xlabel("")
ax.set_ylabel("LLM acceptance")
ax.set_title("Acceptance by difficulty")
ax.set_xticklabels(model_labels, rotation=0)

ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(title="request difficulty", loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
publication_names = {
    "human baseline": "Human baseline",
    "qwen3.6-27b_cadquery-script": "Baseline arm\n(Qwen)",
    "qwen3.6-27b_cadquery-mcp": "Agentic arm\n(Qwen)",
    "gemma4-31b_cadquery-script": "Baseline arm\n(Gemma)",
    "gemma4-31b_cadquery-mcp": "Agentic arm\n(Gemma)",
    "gpt-5.6-luna_cadquery-script": "Baseline arm\n(Luna)",
    "gpt-5.6-luna_cadquery-mcp": "Agentic arm\n(Luna)",
}
publication_model_labels = [
    publication_names.get(user, user)
    for user in plot_users
]

plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 18,
    "legend.title_fontsize": 18,
})

In [ ]:
ax = acceptance_by_difficulty.plot.bar(
    figsize=(14, 7),
    color=difficulty_colors,
    width=0.8,
)

ax.set_ylim(0, 1)
ax.set_xlabel("")
ax.set_ylabel("LLM acceptance")
ax.set_xticklabels(
    publication_model_labels,
    rotation=0,
    ha="center",
)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(title="Request difficulty", loc="upper right")

fig = ax.get_figure()
fig.subplots_adjust(left=0.08, right=0.99, bottom=0.20, top=0.98)
fig.savefig(
    "acceptance_by_request_difficulty.pdf",
    format="pdf",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.01,
    transparent=True,
)
plt.show()

In [ ]:
ax = acceptance_by_modality.plot.bar(
    figsize=(14, 7),
    color=modality_colors,
    width=0.8,
)

ax.set_ylim(0, 1)
ax.set_xlabel("")
ax.set_ylabel("LLM acceptance")
ax.set_xticklabels(
    publication_model_labels,
    rotation=0,
    ha="center",
)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(
    title="Modality",
    loc="upper right",
    bbox_to_anchor=(0.93, 1.0),
)

fig = ax.get_figure()
fig.subplots_adjust(left=0.08, right=0.99, bottom=0.20, top=0.98)
fig.savefig(
    "acceptance_by_input_modality.pdf",
    format="pdf",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.01,
    transparent=True,
)
plt.show()